# 🐱 Kitten Pomodoro — gen lại 6 clip bị cắt khung (MiniMax H3, A100)

Chạy **MiniMax H3** mã nguồn mở (`Comfy-Org/MiniMax-H3`) trong ComfyUI, khung đầu + khung cuối, nền xanh, rồi gọi `video_gen/comfy_batch.py` chạy hàng loạt clip trong `clips.json`.
Prompt được dựng theo định dạng có cấu trúc + timeline giống notebook mẫu (subject_definitions / summary / retention_analysis / detailed_description).

- Runtime → Change runtime type → **A100 GPU** (nên) hoặc **L4** (chạy được nhưng chậm hơn, xem ô cấu hình: tự bật chế độ tiết kiệm VRAM + độ phân giải 896×512).
- Model lưu ở Drive (`ComfyUI_H3_models`) nên các phiên sau không phải tải lại.
- Mặc định 1344×768, 24fps, `quality = "full"` (20 bước). Muốn nhanh hơn: `"turbo"` (LoRA 8-step, 6 bước).

## 1. Kiểm tra GPU

In [ ]:
import subprocess, torch
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], text=True))
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda, "| available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Chưa bật GPU: Runtime > Change runtime type > A100 hoặc L4.")
name = torch.cuda.get_device_name(0)
print("OK:", name) if "A100" in name else print("CHÚ Ý: runtime là", name, "- dùng chế độ GPU yếu (xem ô cấu hình). Có thể mất nhiều thời gian hơn A100.")
!apt-get -qq install -y ffmpeg > /dev/null

## 2. Cấu hình + Google Drive + mã nguồn

- `SOURCE = "github"`: clone repo (cần đã push `video_gen/`). Hoặc `"drive"`: copy `video_gen` lên `MyDrive/kitten_video_gen`.
- `MODEL_VARIANT`: `"int8"` (khuyên dùng, như notebook mẫu) hoặc `"fp8"` nếu int8 lỗi.
- `QUALITY`: `"full"` hoặc `"turbo"`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SOURCE = "github"          # "github" | "drive"
REPO = "https://github.com/DanielQH07/Kitten-Pomodoro.git"
MODEL_VARIANT = "int8"     # "int8" | "fp8"
QUALITY = "full"           # "full" | "turbo"  (turbo: 6 bước, nhanh hơn nhiều, hơi kém nét)
import torch
LOW_VRAM = torch.cuda.get_device_properties(0).total_memory < 30e9    # L4 (24GB) / T4: True; A100: False
RES = "896x512" if LOW_VRAM else ""      # L4: giảm độ phân giải để đủ VRAM; muốn ép: RES = "1344x768"
if LOW_VRAM and QUALITY == "full": print("Gợi ý: GPU yếu nên thử QUALITY = 'turbo' để chạy nhanh hơn.")
RES_ARG = f"--resolution {RES}" if RES else ""
print("LOW_VRAM:", LOW_VRAM, "| RES:", RES or "mặc định 1344x768")

COMFY_ROOT = "/content/ComfyUI"
DRIVE_MODEL_ROOT = "/content/drive/MyDrive/ComfyUI_H3_models"
OUT = "/content/drive/MyDrive/kitten_video_h3_fix"  # kết quả: 1 thư mục, mỗi hành động 1 file <tên>.mp4

import os, shutil
os.makedirs(OUT, exist_ok=True); os.makedirs(DRIVE_MODEL_ROOT, exist_ok=True)
if os.path.exists("/content/video_gen"): shutil.rmtree("/content/video_gen")
if SOURCE == "github":
    !git clone --depth 1 {REPO} /content/_repo
    shutil.copytree("/content/_repo/video_gen", "/content/video_gen")
else:
    shutil.copytree("/content/drive/MyDrive/kitten_video_gen", "/content/video_gen")
!ls /content/video_gen /content/video_gen/refs

## 3. Cài ComfyUI (bản mới nhất, có sẵn node MiniMax H3)

In [ ]:
import os, subprocess, sys
from pathlib import Path
if not Path(COMFY_ROOT).exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", COMFY_ROOT], check=True)
else:
    subprocess.run(["git", "-C", COMFY_ROOT, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY_ROOT}/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"], check=True)
if not Path(COMFY_ROOT, "comfy_extras", "nodes_minimax_h3.py").exists():
    raise RuntimeError("ComfyUI chưa có node MiniMax H3: hãy cập nhật ComfyUI lên bản mới nhất.")
print("ComfyUI sẵn sàng:", COMFY_ROOT)

## 4. Tải model từ Hugging Face (lưu cache ở Drive)
Danh sách nằm trong `clips.json → models_h3local`. File lớn (text encoder Qwen3-VL 32B + DiT), lần đầu tải mất vài chục phút.

In [ ]:
import json
from huggingface_hub import hf_hub_download
cfg = json.load(open("/content/video_gen/clips.json", encoding="utf-8"))
for m in cfg["models_h3local"]:
    if m.get("variant") and m["variant"] != MODEL_VARIANT: continue
    if m.get("optional") and m["optional"] != QUALITY: continue
    target = Path(DRIVE_MODEL_ROOT) / m["dir"] / m["file"]
    if target.exists() and target.stat().st_size > 0:
        print("đã có:", target.name)
    else:
        print("tải:", m["repo"], "/", m["path"])
        target.parent.mkdir(parents=True, exist_ok=True)
        got = hf_hub_download(repo_id=m["repo"], filename=m["path"], local_dir=str(Path(DRIVE_MODEL_ROOT) / "_dl"))
        shutil.move(got, target)
    dest_dir = Path(COMFY_ROOT) / "models" / m["dir"]; dest_dir.mkdir(parents=True, exist_ok=True)
    link = dest_dir / m["file"]
    if link.is_symlink() or link.exists(): link.unlink()
    link.symlink_to(target)
print("xong")

## 5. Bật ComfyUI (chạy nền, không cần ngrok vì runner gọi qua localhost)
Nếu lỗi hết VRAM thử `QUALITY = "turbo"` hoặc thêm `"--lowvram"` vào danh sách tham số.

In [ ]:
import subprocess, time, urllib.request
log = open("/content/comfyui.log", "w")
server = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"] + (["--lowvram"] if LOW_VRAM else []),
    cwd=COMFY_ROOT, stdout=log, stderr=subprocess.STDOUT)
for i in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2); print("ComfyUI sẵn sàng sau", i * 2, "giây"); break
    except Exception:
        time.sleep(2)
else:
    print(open("/content/comfyui.log").read()[-4000:])

## 6. Kiểm tra node + xem danh sách clip (khung/giây theo H3)

In [ ]:
!python /content/video_gen/comfy_batch.py --backend h3local --check
!python /content/video_gen/comfy_batch.py --backend h3local --list

## 7. Chạy thử 1 clip: swat_w (ngắn nhất)
Kết quả lưu thẳng OUT/swat.mp4. Xem ở ô sau rồi mới chạy cả loạt.

In [ ]:
!python /content/video_gen/comfy_batch.py --backend h3local --quality {QUALITY} --variant {MODEL_VARIANT} --out {OUT} --flat --only swat_w {RES_ARG}

In [ ]:
from IPython.display import Video, display
import glob
f = sorted(glob.glob(OUT + "/*.mp4")); print(f)
if f: display(Video(f[-1], embed=True, width=720))

## 8. Chạy 6 clip cần gen lại (batch fix)
Chỉ các clip mà mèo bị mất cảnh vì vượt ra ngoài khung, dùng ảnh tham chiếu "quay xa" (mèo nhỏ 72%, nhiều khoảng trống trên đầu):
- pounce (vồ), reach_up (với cao), swat (vả khi ngồi), land (rơi xuống tiếp đất nhanh), catch_bite (chụp mồi)
- eat: mèo đi tới rồi cúi xuống ăn (đứng nguyên)
- Mỗi clip 1 seed, lưu <tên>.mp4.

In [ ]:
!python /content/video_gen/comfy_batch.py --backend h3local --quality {QUALITY} --variant {MODEL_VARIANT} --out {OUT} --flat --batch fix {RES_ARG}

## 9. Nén thành 1 file zip để tải về
Tải kitten_video_h3_fix.zip trong Drive, giải nén vào video_gen/output/fix/ của dự án rồi chạy
python tools/build_clips.py video_gen/output/fix kitten2/clips

In [ ]:
import shutil; print(shutil.make_archive(OUT, "zip", OUT))

Xem log khi lỗi:

In [ ]:
print(open("/content/comfyui.log", errors="ignore").read()[-12000:])